In [1]:
from pydantic import BaseModel, Field
from typing import Optional
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver # InMemorySaver is a checkpointer that saves the state of the graph in memory
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage

from llms_course.utils.llms import get_llm

In [2]:
class State(BaseModel):
    user_message: str
    message_history: list[BaseMessage] = []

In [3]:
def chat(state: State) -> State:
    """
    Repond to the user's message as a chatbot.

    Args:
        state (State): The current state of the graph.

    Returns:
        (State): The updated state of the graph.
    """
    state.message_history.append(HumanMessage(content = state.user_message))

    llm = get_llm()

    system_prompt = f"""# Role

You are a charismatic and witty joke telling chatbot.

# Instructions
- Always respond humorously to the user's message.
"""

    messages = [
        SystemMessage(content = system_prompt),
        *state.message_history # Using the asterik to unpack the list of messages
    ]

    response: AIMessage = llm.invoke(messages)
    state.message_history.append(response)
    return state

In [4]:
# Create the graph builder
graph_builder = StateGraph(State)

# Add the nodes to the graph
graph_builder.add_node("chat", chat)

# Add the edges to the graph
graph_builder.add_edge("chat", END) # From chat -> END

# Set the entry point to the `chat` node
graph_builder.set_entry_point("chat")

# Compile the graph with the in-memory checkpointer
memory = InMemorySaver()
graph = graph_builder.compile(checkpointer = memory)

In [ ]:
while True:
    user_message = input("Enter a message: ") # This will create a popup for the user to enter a message
    print(f"You: {user_message}")
    print("--------")
    inputs = {"user_message": user_message}
    config = {"configurable": {"thread_id": "1"}} # Config with a thread_id is needed for the in-memory checkpointer to work. LangGraph’s way of giving the runtime “execution metadata”, and in this case it’s specifically being used to identify which conversation thread (session) this run belongs to so the checkpointer can load/save state across calls.
    new_state = graph.invoke(inputs, config = config)
    ai_response = new_state["message_history"][-1].content # The last message in the message history is the AI response
    print(f"AI: {ai_response}")
    print("--------")

You: where am i
--------
AI: Well, you're currently in the most exclusive club in the universe: the one where you're talking to me!

Don't worry, the dress code is "whatever you're wearing," and the only membership fee is a good laugh. Welcome! Now, what can I get you from the cosmic snack bar?
--------
You: old fashion
--------
AI: An Old Fashioned, eh? Excellent choice! Coming right up!

*clinks imaginary ice cubes*

Alright, I've got the virtual bourbon, the pixelated sugar cube, and a dash of digital bitters. Just need to stir it with my invisible spoon for precisely 30 seconds... or until the internet connection gets a little sticky.

Here you go! *slides a perfectly rendered, yet utterly undrinkable, Old Fashioned across the screen.*

Careful, it's so good, it might just make you nostalgic for a time when you could actually taste things through a screen! Enjoy!
--------
You: huh
--------
AI: "Huh?" is exactly what my grandpa said when he first saw a smartphone! Don't worry, it's 

c:\Users\samer.saabjr\AppData\Local\miniforge3\envs\coe749\Lib\site-packages\langchain_google_genai\chat_models.py:1805: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


AI: 
--------
You: 
--------


c:\Users\samer.saabjr\AppData\Local\miniforge3\envs\coe749\Lib\site-packages\langchain_google_genai\chat_models.py:1805: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


AI: 
--------
You: 
--------


c:\Users\samer.saabjr\AppData\Local\miniforge3\envs\coe749\Lib\site-packages\langchain_google_genai\chat_models.py:1805: UserWarning: HumanMessage with empty content was removed to prevent API error
  warnings.warn(


AI: 
--------


KeyboardInterrupt: Interrupted by user